# Binary Image Classifier — Cats vs Dogs

Train a binary classifier to distinguish between **cat** and **dog** images using a convolutional neural network (CNN) in PyTorch.

This project introduces:
- Loading images from folders with `torchvision.datasets.ImageFolder`
- Image preprocessing and data augmentation with `transforms`
- Building a CNN for binary classification
- A full training loop with a **validation split**, accuracy tracking, and metric plots
- Running inference on a single new image

**Goal:** classify an input image as either a cat or a dog.

## 1. Imports and device setup

In [ ]:
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

## 2. Dataset layout

`ImageFolder` expects one subfolder per class. Arrange your images like this:

```
data/
  cats_vs_dogs/
    train/
      cat/   -> cat.0.jpg, cat.1.jpg, ...
      dog/   -> dog.0.jpg, dog.1.jpg, ...
```

A popular source is the [Kaggle Dogs vs. Cats dataset](https://www.kaggle.com/c/dogs-vs-cats/data). Download it, then sort the images into `cat/` and `dog/` subfolders under `train/`.

The class-to-index mapping is assigned alphabetically, so `cat` -> 0 and `dog` -> 1. We treat the model's single sigmoid output as **P(dog)**.

In [ ]:
# Point this at your training data root (the folder containing cat/ and dog/)
DATA_DIR = os.path.join("..", "data", "cats_vs_dogs", "train")

IMG_SIZE = 128
BATCH_SIZE = 32

## 3. Transforms (preprocessing + augmentation)

We use light augmentation on the training set (random flips/rotations) to reduce overfitting, and plain resize + normalize for validation. Normalizing all three RGB channels keeps pixel values roughly in `[-1, 1]`.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

## 4. Load data and create train/validation split

We load the full dataset once, split it 80/20 into training and validation subsets, and then apply the appropriate transform to each subset.

In [ ]:
# Base dataset (transform set per-split below)
full_dataset = datasets.ImageFolder(root=DATA_DIR)
print(f"Classes: {full_dataset.classes}  ->  mapping {full_dataset.class_to_idx}")
print(f"Total images: {len(full_dataset)}")

val_frac = 0.2
val_size = int(len(full_dataset) * val_frac)
train_size = len(full_dataset) - val_size
train_subset, val_subset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

# Apply train-time augmentation to the training split, plain transform to validation.
# (Subsets share the underlying dataset, so we wrap them to override the transform.)
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset.dataset.samples[self.subset.indices[idx]]
        from PIL import Image
        image = Image.open(img).convert("RGB")
        return self.transform(image), label

train_dataset = TransformedSubset(train_subset, train_transform)
val_dataset = TransformedSubset(val_subset, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

## 5. Visualize a batch

A quick sanity check that images load and labels line up. We un-normalize for display.

In [ ]:
def imshow(ax, img_tensor):
    img = img_tensor * 0.5 + 0.5  # un-normalize
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.axis("off")

images, labels = next(iter(train_loader))
classes = full_dataset.classes

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for i, ax in enumerate(axes):
    imshow(ax, images[i])
    ax.set_title(classes[labels[i]])
plt.tight_layout()
plt.show()

## 6. Define the CNN

Two convolution + pooling blocks, then a fully connected head with a single sigmoid output for binary classification.

Instead of hardcoding the flattened feature size, we compute it dynamically from a dummy forward pass. This keeps the model correct even if you change `IMG_SIZE`.

In [ ]:
class CatDogCNN(nn.Module):
    def __init__(self, img_size=128):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Compute the flattened size dynamically so it adapts to img_size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size)
            n_flat = self._features(dummy).view(1, -1).shape[1]

        self.fc1 = nn.Linear(n_flat, 1)

    def _features(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        return x

    def forward(self, x):
        x = self._features(x)
        x = x.view(x.size(0), -1)        # flatten, keep batch dim
        return torch.sigmoid(self.fc1(x))

model = CatDogCNN(img_size=IMG_SIZE).to(device)
print(model)

## 7. Loss and optimizer

Binary cross-entropy loss (the model already applies sigmoid), optimized with Adam.

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 8. Training loop with validation

Each epoch we train on the training split, then evaluate on the validation split without computing gradients. We track loss and accuracy for both so we can plot learning curves and watch for overfitting.

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    torch.set_grad_enabled(train)
    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)  # shape [B, 1]

        outputs = model(images)
        loss = criterion(outputs, labels)

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = (outputs >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += images.size(0)

    torch.set_grad_enabled(True)
    return total_loss / total, correct / total


EPOCHS = 5
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch + 1}/{EPOCHS} | "
          f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f}")

## 9. Plot learning curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.set_xlabel("epoch")
ax1.legend()

ax2.plot(history["train_acc"], label="train")
ax2.plot(history["val_acc"], label="val")
ax2.set_title("Accuracy")
ax2.set_xlabel("epoch")
ax2.legend()

plt.tight_layout()
plt.show()

## 10. Save and load the model

Persist the trained weights so you can reuse them without retraining.

In [ ]:
SAVE_PATH = "catdog_cnn.pth"
torch.save(model.state_dict(), SAVE_PATH)
print(f"Saved weights to {SAVE_PATH}")

# To reload later:
# model = CatDogCNN(img_size=IMG_SIZE).to(device)
# model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
# model.eval()

## 11. Inference on a single image

Load one image, apply the evaluation transform, and interpret the sigmoid output. Recall the output is **P(dog)**, so a value `>= 0.5` means dog.

In [ ]:
from PIL import Image

def predict_image(path):
    image = Image.open(path).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(device)  # add batch dim
    model.eval()
    with torch.no_grad():
        prob_dog = model(tensor).item()
    label = "dog" if prob_dog >= 0.5 else "cat"
    confidence = prob_dog if label == "dog" else 1 - prob_dog
    return label, confidence, image

# Example usage (point to any image file):
# label, conf, img = predict_image("../images/spoon.jpeg")
# plt.imshow(img); plt.axis("off")
# plt.title(f"Prediction: {label} ({conf:.1%} confidence)")
# plt.show()

## Summary & next steps

- `ImageFolder` turns a folder-per-class layout into a labeled dataset.
- A small CNN (conv → ReLU → pool, twice) plus a sigmoid head handles binary image classification.
- A validation split and accuracy tracking show whether the model generalizes or overfits.

**Ways to improve accuracy:**
- Add more conv blocks or batch normalization
- Use transfer learning (e.g. a pretrained `torchvision.models.resnet18`) — usually a big jump for small datasets
- Add dropout and stronger augmentation to fight overfitting
- Train for more epochs with a learning-rate scheduler